In [1]:
import pandas as pd
import os

In [2]:
# Debugging Configurations
DEBUG = False # Set to True for quick test
DEBUG_FOVS_PER_GROUP = 10
SAVE_RAW_RULES = True

# Path Configuration
DATA_DIR = 'data/'
MIBI_GUT_DIR_PATH = DATA_DIR + 'MIBIGutCsv/'

RESULTS_BASE_DIR = 'results/'
if DEBUG:
    RESULTS_DIR = RESULTS_BASE_DIR + 'debug_run/'
else:
    RESULTS_DIR = RESULTS_BASE_DIR + 'full_run/'

RESULTS_DATA_DIR = RESULTS_DIR + 'data/'
RESULTS_PLOTS_DIR = RESULTS_DIR + 'plots/'

RESULT_EXPLORATION_DIR = 'result_exploration/'
CONSENSUS_RESULTS_EXPLORATION_DIR = RESULTS_DATA_DIR + '/consensus_tables/'

RESULTS_ML_DIR = RESULTS_DATA_DIR + 'ml_refined_robust_benchmarks/'
RESULTS_ML_DATA_DIR = RESULTS_ML_DIR + 'data/'
RESULTS_ML_PLOTS_DIR = RESULTS_ML_DIR + 'plots/'

RESULTS_SIMPLE_STATS_DIR = RESULTS_DATA_DIR + 'simple_stats_benchmarks/'
RESULTS_SIMPLE_STATS_DATA_DIR = RESULTS_SIMPLE_STATS_DIR + 'data/'
RESULTS_SIMPLE_STATS_PLOTS_DIR = RESULTS_SIMPLE_STATS_DIR + 'plots/'

RESULTS_CLINICAL_CORRELATION_PLOTS_DIR = RESULTS_PLOTS_DIR + 'clinical_correlation_report/'

# Param Configuration
MIN_P_VALUE = 0.05


In [3]:
fov_df = pd.read_csv("../" + MIBI_GUT_DIR_PATH + 'fovs_metadata.csv')
print(fov_df[fov_df["FOV"].str.startswith("S_")].head())


            FOV Patient                     Cohort  Size [um]
242  S_01_FOV_1    S_01  Duodenum_Cohort_Control_2        800
243  S_01_FOV_2    S_01  Duodenum_Cohort_Control_2        800
244  S_01_FOV_3    S_01  Duodenum_Cohort_Control_2        800
245  S_01_FOV_4    S_01  Duodenum_Cohort_Control_2        800
246  S_02_FOV_1    S_02  Duodenum_Cohort_Control_2        800


In [4]:
print(fov_df.columns)

Index(['FOV', 'Patient', 'Cohort', 'Size [um]'], dtype='str')


In [5]:
biopsy_df = pd.read_csv("../" + MIBI_GUT_DIR_PATH + 'biopsy_metadata.csv')
print(biopsy_df.columns)

Index(['Biopsy_ID', 'Patient_ID', 'Age', 'Date of biopsy',
       'Immunosuppressants given more than 48h pre-biopsey', 'Gender',
       'Diagnosis', 'Date of transplant', 'Donor gender', 'Donor type',
       'Conditioning', 'Conditioning type', 'ATG', 'PTCy', 'GVHD Prophylaxis',
       'Grade GVHD', 'Pathological stage', 'GI stage', 'liver stage',
       'skin stage', 'Cortico Response', 'Localization',
       'Survival at follow-up', 'DATE Follow-up', 'Cause of death',
       'Days follow-up from transplant', 'Days after Transplant',
       'Days after Transplant grouped', 'Clinical score', 'Pathological score',
       'Donor-Host Sex', 'Chimerism WB (% Donor)', 'Date of chimerism test',
       'd_chimerism - d_biopsy', 'Lymphocyte count [K cells/mcl]',
       'Date of blood count (closest to biopsy)', 'd_blood-d_Tx',
       'Total body irradiation', 'Ovelap syndrome', 'Virus in biopsy'],
      dtype='str')


In [6]:
# print unique values for columns in fov_df when the number of uniques < 50
for col in fov_df.columns:
    n_unique = fov_df[col].nunique(dropna=False)
    if n_unique < 50:
        vals = fov_df[col].unique()
        try:
            vals_sorted = sorted(vals, key=lambda x: (str(type(x)), str(x)))
        except Exception:
            vals_sorted = list(vals)
        print(f"{col} ({n_unique} unique):")
        print(vals_sorted)
        print()

Cohort (4 unique):
['Cohort_GVHD', 'Colon_Cohort_Control', 'Duodenum_Cohort_Control_1', 'Duodenum_Cohort_Control_2']

Size [um] (2 unique):
[np.int64(400), np.int64(800)]



In [7]:
# print unique values for columns in fov_df when the number of uniques < 50
for col in biopsy_df.columns:
    n_unique = biopsy_df[col].nunique(dropna=False)
    if n_unique < 100:
        vals = biopsy_df[col].unique()
        try:
            vals_sorted = sorted(vals, key=lambda x: (str(type(x)), str(x)))
        except Exception:
            vals_sorted = list(vals)
        print(f"{col} ({n_unique} unique):")
        print(vals_sorted)
        print()

Biopsy_ID (59 unique):
['GVHD_01', 'GVHD_02', 'GVHD_03', 'GVHD_04', 'GVHD_05', 'GVHD_06', 'GVHD_07', 'GVHD_08', 'GVHD_09', 'GVHD_10', 'GVHD_11', 'GVHD_12', 'GVHD_13', 'GVHD_14', 'GVHD_15', 'GVHD_16', 'GVHD_17', 'GVHD_18', 'GVHD_19', 'GVHD_20', 'GVHD_21', 'GVHD_22', 'GVHD_23', 'GVHD_24', 'GVHD_25', 'GVHD_26', 'GVHD_27', 'GVHD_28', 'GVHD_29', 'GVHD_30', 'GVHD_31', 'GVHD_32', 'GVHD_33', 'GVHD_34', 'GVHD_35', 'GVHD_36', 'GVHD_37', 'GVHD_38', 'GVHD_39', 'GVHD_40', 'GVHD_41', 'GVHD_42', 'GVHD_43', 'GVHD_44', 'GVHD_45', 'GVHD_47', 'GVHD_48', 'GVHD_49', 'GVHD_50', 'GVHD_51', 'GVHD_52', 'GVHD_53', 'GVHD_54', 'GVHD_55', 'GVHD_56', 'GVHD_57', 'GVHD_58', 'GVHD_59', 'GVHD_60']

Patient_ID (57 unique):
['P_01', 'P_02', 'P_03', 'P_04', 'P_05', 'P_06', 'P_07', 'P_08', 'P_09', 'P_10', 'P_11', 'P_12', 'P_13', 'P_14', 'P_15', 'P_16', 'P_17', 'P_18', 'P_19', 'P_20', 'P_21', 'P_22', 'P_23', 'P_24', 'P_25', 'P_26', 'P_27', 'P_28', 'P_29', 'P_30', 'P_31', 'P_32', 'P_33', 'P_34', 'P_35', 'P_36', 'P_37', 'P_38

In [8]:
cell_df = pd.read_csv("../" + MIBI_GUT_DIR_PATH + 'cell_table.csv')
print(cell_df.columns)

Index(['fov', 'cell_ID', 'cell type', 'population', 'in_CryptVilli',
       'in_BrunnerGland', 'in_SMV', 'in_Muscle', 'in_LP', 'in_Submucosa',
       'in_Follicle', 'in_Lumen', 'CD103', 'CD14', 'CD15', 'CD163', 'CD20',
       'CD206', 'CD209', 'CD3', 'CD31', 'CD38', 'CD4', 'CD45', 'CD45RA',
       'CD45RO', 'CD56', 'CD68', 'CD69', 'CD8', 'CHGA', 'Calprotectin',
       'Collagen', 'ECAD', 'FOXP3', 'GZMB', 'HLA1', 'HLADRDPDQ', 'IDO1', 'IgA',
       'Keratin', 'Ki67', 'Lysozyme', 'Mucin', 'NAKATPASE', 'PD1', 'PDL1',
       'SMA', 'TCF', 'Tryptase', 'VIM', 'dsDNA', 'Area', 'centroid_x',
       'centroid_y'],
      dtype='str')


In [9]:
# print unique values for columns in fov_df when the number of uniques < 50
for col in cell_df.columns:
    n_unique = cell_df[col].nunique(dropna=False)
    if n_unique < 50:
        vals = cell_df[col].unique()
        try:
            vals_sorted = sorted(vals, key=lambda x: (str(type(x)), str(x)))
        except Exception:
            vals_sorted = list(vals)
        print(f"{col} ({n_unique} unique):")
        print(vals_sorted)
        print()

cell type (32 unique):
['APC', 'Bcell', 'BrunnerGland', 'CD3T', 'CD4T', 'CD8T', 'DC', 'Endocrine', 'Endothelial', 'Epithelial', 'Fibroblast', 'FibroticEpithelial', 'Goblet', 'ImmuneOther', 'ImmuneOther_CD45RA', 'Macrophage', 'Macrophage_Calp', 'Mast', 'Mesenchymal_VIM', 'Monocyte', 'Muscle', 'NK', 'Neuroendocrine', 'Neuron', 'Neutrophil', 'Neutrophil_CD15', 'Paneth', 'Plasma', 'Plasma_CD38', 'SMV', 'Treg', 'Unidentified']

population (4 unique):
[nan, 'Epithel', 'Immune', 'Stroma']

in_CryptVilli (2 unique):
[np.False_, np.True_]

in_BrunnerGland (2 unique):
[np.False_, np.True_]

in_SMV (2 unique):
[np.False_, np.True_]

in_Muscle (2 unique):
[np.False_, np.True_]

in_LP (2 unique):
[np.False_, np.True_]

in_Submucosa (2 unique):
[np.False_, np.True_]

in_Follicle (2 unique):
[np.False_, np.True_]

in_Lumen (2 unique):
[np.False_, np.True_]



In [13]:
cells_per_fov = cell_df.groupby('fov').size().reset_index(name='cell_count')
print(cells_per_fov.sort_values(by='cell_count', ascending=False).to_string(index=False))

                   fov  cell_count
Control_Colon_04_FOV_6        7983
Control_Colon_04_FOV_5        7395
            S_01_FOV_3        6850
            S_03_FOV_3        6690
            S_01_FOV_2        6565
            S_02_FOV_3        6260
            S_02_FOV_1        6163
            S_01_FOV_4        6072
Control_Colon_03_FOV_6        6036
            S_03_FOV_7        5986
            S_02_FOV_4        5916
            S_03_FOV_8        5882
            S_02_FOV_2        5806
Control_Colon_03_FOV_5        5717
Control_Colon_02_FOV_6        5709
Control_Colon_02_FOV_5        5691
Control_Colon_01_FOV_7        5504
            S_03_FOV_2        5497
            S_03_FOV_4        5350
Control_Colon_01_FOV_6        5271
            S_03_FOV_1        5247
            S_02_FOV_5        5177
            S_03_FOV_5        5168
           S_03_FOV_10        5123
            S_01_FOV_1        5115
            S_03_FOV_6        4798
            S_03_FOV_9        4552
            S_04_FOV

In [ ]:
# Check tissue region overlaps
TISSUE_COLS = ['in_CryptVilli', 'in_BrunnerGland', 'in_SMV', 'in_Muscle',
                'in_LP', 'in_Submucosa', 'in_Follicle', 'in_Lumen']

# Visualize the distribution of cells in multiple regions
print(f"\nBreakdown by region count:")
for num_regions in sorted(cell_df['num_regions'].unique()):
    count = (cell_df['num_regions'] == num_regions).sum()
    pct = 100 * count / len(cell_df)
    print(f"  {int(num_regions)} region(s): {count:,} cells ({pct:.1f}%)")


Breakdown by region count:
  0 region(s): 80,297 cells (11.3%)
  1 region(s): 594,258 cells (83.3%)
  2 region(s): 38,621 cells (5.4%)
  3 region(s): 195 cells (0.0%)
  4 region(s): 1 cells (0.0%)


: 